In [6]:
import sys
sys.path.append('/root/LLMs/CosyVoice')
sys.path.append('/root/LLMs/CosyVoice/third_party/Matcha-TTS')


from hyperpyyaml import load_hyperpyyaml
from cosyvoice.cli.frontend import CosyVoiceFrontEnd
import os
from cosyvoice.cli.model import CosyVoice2Model
from cosyvoice.utils.file_utils import load_wav
import torchaudio


model_dir = "/root/LLMs/CosyVoice/pretrained_models/CosyVoice2-0.5B"
with open('{}/cosyvoice.yaml'.format(model_dir), 'r') as f:
            configs = load_hyperpyyaml(f, overrides={'qwen_pretrain_path': os.path.join(model_dir, 'CosyVoice-BlankEN')})
frontend = CosyVoiceFrontEnd(configs['get_tokenizer'],
                                          configs['feat_extractor'],
                                          '{}/campplus.onnx'.format(model_dir),
                                          '{}/speech_tokenizer_v2.onnx'.format(model_dir),
                                          '{}/spk2info.pt'.format(model_dir),
                                          configs['allowed_special'])

model = CosyVoice2Model(configs['llm'], configs['flow'], configs['hift'], fp16=False)

model.load('{}/llm.pt'.format(model_dir),
           '{}/flow.pt'.format(model_dir),
           '{}/hift.pt'.format(model_dir))


failed to import ttsfrd, use WeTextProcessing instead


/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)
2025-01-06 21:32:41,921 INFO input frame rate=25
/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:28: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")
/root/LLMs/CosyVoice/cosyvoice/dataset/processor.py:24: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend('soundfile')
2025-01-06 

In [1]:
import torch
import torchaudio
import whisper
import onnxruntime
import numpy as np

def extract_speech_token(audio_path, onnx_model_path):
    """
    提取单个音频文件的 speech token
    :param audio_path: 音频文件路径
    :param onnx_model_path: ONNX 模型路径
    :return: speech token (numpy array)
    """
    # 加载音频文件
    audio, sample_rate = torchaudio.load(audio_path)

    # 如果采样率不是16000，则进行重采样
    if sample_rate != 16000:
        audio = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)(audio)

    # 如果音频长度超过30秒，则返回空列表
    if audio.shape[1] / 16000 > 30:
        print(f'音频 {audio_path} 超过30秒，不支持提取语音特征')
        return []

    # 计算音频的梅尔频谱图
    feat = whisper.log_mel_spectrogram(audio, n_mels=128)

    # 配置ONNX Runtime会话
    option = onnxruntime.SessionOptions()
    option.graph_optimization_level = onnxruntime.GraphOptimizationLevel.ORT_ENABLE_ALL
    ort_session = onnxruntime.InferenceSession(onnx_model_path, sess_options=option)

    # 获取输入名称
    input_name = ort_session.get_inputs()[0].name
    input_length_name = ort_session.get_inputs()[1].name

    # 使用ONNX模型进行推理
    outputs = ort_session.run(None, {
        input_name: feat.detach().cpu().numpy(),
        input_length_name: np.array([feat.shape[2]], dtype=np.int32)
    })

    # 返回 speech token
    return outputs[0].flatten()


In [2]:
import torch
import sys

sys.path.append('/root/LLMs/CosyVoice/third_party/Matcha-TTS')
from matcha.utils.audio import mel_spectrogram

# sampling_rate = 22050

class Feat_extractor:

    def __init__(self, n_fft: int = 1920, num_mels: int = 80, sampling_rate: int = 24000, 
                 hop_size: int = 480, win_size: int = 1920, fmin: int = 0, fmax: int = 8000, 
                 center: bool = False, device: str = 'cuda'):
        self.device = torch.device(device)
        self.feat_extractor = lambda x: mel_spectrogram(
            x, n_fft=n_fft, num_mels=num_mels, sampling_rate=sampling_rate, 
            hop_size=hop_size, win_size=win_size, fmin=fmin, fmax=fmax, center=center
        )

    def __call__(self, speech):
        speech_feat = self.feat_extractor(speech).squeeze(dim=0).transpose(0, 1).to(self.device)
        speech_feat = speech_feat.unsqueeze(dim=0)
        speech_feat_len = torch.tensor([speech_feat.shape[1]], dtype=torch.int32).to(self.device)
        return speech_feat, speech_feat_len


/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch
import torchaudio
import torchaudio.compliance.kaldi as kaldi
import onnxruntime
import numpy as np

def extract_embedding(audio_path, onnx_model_path):
    """
    提取单个音频文件的 embedding
    :param audio_path: 音频文件路径
    :param onnx_model_path: ONNX 模型路径
    :return: embedding (numpy array)
    """
    # 加载音频文件
    audio, sample_rate = torchaudio.load(audio_path)

    # 如果采样率不是16000，则进行重采样
    if sample_rate != 16000:
        audio = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)(audio)

    # 计算音频的FBank特征
    feat = kaldi.fbank(
        audio,
        num_mel_bins=80,  # 梅尔频带数
        dither=0,         # 禁用抖动
        sample_frequency=16000
    )

    # 特征归一化
    feat = feat - feat.mean(dim=0, keepdim=True)

    # 配置ONNX Runtime会话
    option = onnxruntime.SessionOptions()
    option.graph_optimization_level = onnxruntime.GraphOptimizationLevel.ORT_ENABLE_ALL
    ort_session = onnxruntime.InferenceSession(onnx_model_path, sess_options=option)

    # 使用ONNX模型进行推理
    input_name = ort_session.get_inputs()[0].name
    embedding = ort_session.run(None, {input_name: feat.unsqueeze(dim=0).cpu().numpy()})[0]

    # 返回 embedding
    return embedding.flatten()

In [16]:
import sys
sys.path.append('/root/LLMs/CosyVoice')

import uuid
from cosyvoice.utils.file_utils import load_wav
import torchaudio

voice_tokenizer_path = '/root/LLMs/CosyVoice/pretrained_models/CosyVoice2-0.5B/speech_tokenizer_v2.onnx'
embedding_path = '/root/LLMs/CosyVoice/pretrained_models/CosyVoice2-0.5B/campplus.onnx'

resample_rate = 24000

input_speech_16k_path = '/root/LLMs/CosyVoice/data/zero_shot_prompt.wav'
input_speech_16k = load_wav(input_speech_16k_path, 16000)
# input_text = '希望你以后能够做的比我还好呦。'

prompt_speech_16k_path = '/root/LLMs/CosyVoice/data/ZH_B00000_S00000_W000000.mp3'
prompt_speech_16k = load_wav(prompt_speech_16k_path, 16000)
# input_text = '能源风口，一路护航，你们好，我是张航本航。'

# 提取语音 token
input_token = extract_speech_token(input_speech_16k_path, voice_tokenizer_path)
input_token = torch.tensor(input_token).unsqueeze(0)

# 提取 prompt 语音 token
prompt_token = extract_speech_token(prompt_speech_16k_path, voice_tokenizer_path)
prompt_token = torch.tensor(prompt_token).unsqueeze(0)


# 提取 mel 特征
front_end = Feat_extractor()

input_speech_resample = torchaudio.transforms.Resample(orig_freq=16000, new_freq=resample_rate)(input_speech_16k)
input_feat, input_feat_len = front_end(input_speech_resample)

# 提取 prompt mel 特征
prompt_speech_resample = torchaudio.transforms.Resample(orig_freq=16000, new_freq=resample_rate)(prompt_speech_16k)
prompt_feat, prompt_feat_len = front_end(prompt_speech_resample)

# 提取 speaker embedding
embedding = extract_embedding(input_speech_16k_path, embedding_path)
embedding =  torch.tensor(embedding).unsqueeze(0).to('cuda')

# 提取 prompt embedding
prompt_embedding = extract_embedding(prompt_speech_16k_path, embedding_path)
prompt_embedding =  torch.tensor(prompt_embedding).unsqueeze(0).to('cuda')

In [ ]:
import torch
import IPython.display as ipd

this_uuid = str(uuid.uuid1())
model.hift_cache_dict[this_uuid] = None

prompt_token1=torch.zeros(1, 0, dtype=torch.int32)
prompt_feat=torch.zeros(1, 0, 80)
embedding=torch.zeros(1, 192)

this_tts_speech = model.token2wav(token=prompt_token,
                                 prompt_token=prompt_token1,#speech_token,
                                 prompt_feat=prompt_feat,#speech_feat,
                                 embedding=embedding,
                                 uuid=this_uuid,
                                 token_offset=0,
                                 finalize=True,
                                 speed=1)

# prompt_feat=torch.zeros(1, 0, 80), embedding=torch.zeros(1, 192)
# 假设 audio_data 是你的音频数据，sr 是采样率
# audio_data = np.array(new_dataset['mp3'][0]['array'])
sr = 24000
tts_speech = this_tts_speech.cpu()
# 在 Jupyter Notebook 中播放音频
ipd.Audio(tts_speech, rate=sr)

GLM voice

In [1]:
import re

def extract_numbers_from_string(s):
    # 使用正则表达式找到所有符合数字模式的序列，包括负数和小数
    number_pattern = r'-?\d+\.?\d*'
    matches = re.findall(number_pattern, s)
    
    # 将找到的字符串格式的数字转换为适当的数值类型（整数或浮点数）
    numbers = []
    for match in matches:
        try:
            # 如果字符串包含小数点，则转换为浮点数
            if '.' in match:
                numbers.append(float(match))
            else:
                # 否则转换为整数
                numbers.append(int(match))
        except ValueError:
            # 如果转换失败，忽略该匹配项
            continue
    
    return numbers

In [4]:
audio_tokens = "北京啊，那可是个充满活力和魅力的地方！<|audio_10815|><|audio_14121|><|audio_14268|><|audio_15298|><|audio_7506|><|audio_15400|><|audio_13364|><|audio_14248|><|audio_14445|><|audio_15428|><|audio_10640|><|audio_557|><|audio_3322|><|audio_10973|><|audio_2968|><|audio_13089|><|audio_1213|><|audio_12153|><|audio_8995|><|audio_5838|><|audio_13549|><|audio_14720|><|audio_4623|><|audio_6648|><|audio_8160|><|audio_10148|>无论是历史悠久的前门、故宫，还是现代化的三里屯、<|audio_1329|><|audio_10023|><|audio_3148|><|audio_8336|><|audio_13255|><|audio_5422|><|audio_6353|><|audio_13933|><|audio_5554|><|audio_4781|><|audio_6237|><|audio_11993|><|audio_14934|><|audio_1733|><|audio_1504|><|audio_6914|><|audio_13800|><|audio_12714|><|audio_16007|><|audio_12041|><|audio_12890|><|audio_5296|><|audio_10420|><|audio_10693|><|audio_5900|><|audio_8227|>国贸，都能让你流连忘返。而且北京的美食<|audio_8790|><|audio_5889|><|audio_11346|><|audio_10813|><|audio_362|><|audio_14166|><|audio_12463|><|audio_692|><|audio_12683|><|audio_6237|><|audio_624|><|audio_15726|><|audio_13558|><|audio_7856|><|audio_11684|><|audio_13664|><|audio_15528|><|audio_8206|><|audio_15287|><|audio_12959|><|audio_4230|><|audio_8884|><|audio_11734|><|audio_13307|><|audio_9614|><|audio_13612|>也是全国各地的风味都有，绝对能让你大饱口福<|audio_11495|><|audio_11024|><|audio_1769|><|audio_15069|><|audio_3822|><|audio_9716|><|audio_4401|><|audio_317|><|audio_13089|><|audio_9435|><|audio_10391|><|audio_9947|><|audio_1511|><|audio_4804|><|audio_15775|><|audio_6131|><|audio_4421|><|audio_1474|><|audio_5674|><|audio_11342|><|audio_12594|><|audio_3032|><|audio_14901|><|audio_14399|><|audio_12714|><|audio_15526|>！<|audio_14458|><|audio_9102|><|audio_12463|><|audio_4113|><|audio_7553|><|audio_13964|><|audio_15569|><|audio_5200|><|audio_485|><|audio_13176|><|audio_4336|><|audio_4357|><|audio_1392|><|audio_7746|><|audio_14593|><|audio_1322|><|audio_13627|><|audio_10036|><|audio_15470|><|audio_12714|><|audio_16297|><|audio_5913|><|audio_11783|><|audio_12123|><|audio_1502|><|audio_11585|><|audio_9546|><|audio_1871|><|audio_5764|><|audio_15935|><|audio_10660|><|audio_12447|><|audio_292|><|audio_12057|><|audio_8886|><|audio_10293|><|audio_120|><|audio_16059|><|audio_11151|><|audio_13706|><|audio_3793|><|audio_9306|><|audio_3412|><|audio_2771|><|audio_9653|><|audio_3954|><|audio_7260|><|audio_15662|><|audio_13737|><|audio_12112|><|audio_3374|><|audio_14720|><|audio_5389|><|audio_4143|><|audio_41|><|audio_11806|><|audio_8809|><|audio_14383|><|audio_6317|><|audio_9247|><|audio_7384|><|audio_1789|><|audio_15848|><|audio_6105|><|audio_4726|><|audio_13162|><|audio_11002|><|audio_15444|><|audio_3260|><|audio_379|><|audio_2372|><|audio_15120|><|audio_8406|><|audio_14050|><|audio_4225|><|audio_1280|><|audio_9875|><|audio_5354|><|audio_10192|><|audio_1051|><|audio_12041|><|audio_15724|>"

In [5]:
audio_tokens = extract_numbers_from_string(audio_tokens)

In [1]:
import sys
import os
sys.path.append('/root/LLMs/GLM-4-Voice')

import torch
from flow_inference import AudioDecoder
import os.path


flow_path="/root/LLMs/GLM-4-Voice/THUDM/glm-4-voice-decoder"
tokenizer_path="/root/LLMs/GLM-4-Voice/THUDM/glm-4-voice-tokenizer"

flow_config = os.path.join(flow_path, "config.yaml")
flow_checkpoint = os.path.join(flow_path, "flow.pt")
hift_checkpoint = os.path.join(flow_path, "hift.pt")
device = "cuda"
audio_decoder: AudioDecoder = None
whisper_model, feature_extractor = None, None

sys.path.insert(0, "/root/LLMs/GLM-4-Voice/cosyvoice")
sys.path.insert(0, "/root/LLMs/CosyVoice/third_party/Matcha-TTS")

# Flow & Hift
audio_decoder = AudioDecoder(config_path=flow_config,
                             flow_ckpt_path=flow_checkpoint,
                             hift_ckpt_path=hift_checkpoint,
                             device=device,)

import uuid
# 生成唯一标识符
this_uuid = str(uuid.uuid4())

/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/torch/_jit_internal.py:739: FutureWarning: ignore(True) has been deprecated. TorchScript will now drop the function call on compilation. Use torch.jit.unused now. {}
  warnings.warn(
/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)
/root/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:28: UserWarning: torch.nn.utils.weigh

In [19]:
from speech_tokenizer.modeling_whisper import WhisperVQEncoder
from transformers import WhisperFeatureExtractor
from speech_tokenizer.utils import extract_speech_token

# 提取音频中的 token
input_speech_16k_path = ['/root/LLMs/CosyVoice/data/ZH_B00000_S00000_W000000.mp3']

# Speech tokenizer
whisper_model = (WhisperVQEncoder.from_pretrained(tokenizer_path).eval().to(device))

feature_extractor = WhisperFeatureExtractor.from_pretrained(tokenizer_path)

# 提取音频中的 token
audio_tokens = extract_speech_token(whisper_model, feature_extractor, input_speech_16k_path)[0]
audio_tokens = torch.tensor(audio_tokens).unsqueeze(0).to('cuda')

In [27]:
import sys
sys.path.append('/root/LLMs/CosyVoice')

# import uuid
from cosyvoice.utils.file_utils import load_wav
import torchaudio

voice_tokenizer_path = '/root/LLMs/CosyVoice/pretrained_models/CosyVoice2-0.5B/speech_tokenizer_v2.onnx'
embedding_path = '/root/LLMs/CosyVoice/pretrained_models/CosyVoice2-0.5B/campplus.onnx'
# this_uuid = str(uuid.uuid1())
# model.hift_cache_dict[this_uuid] = None

resample_rate = 22050

input_speech_16k_path = '/root/LLMs/CosyVoice/data/ZH_B00000_S00000_W000000.mp3'
input_speech_16k = load_wav(input_speech_16k_path, 16000)
# input_text = '希望你以后能够做的比我还好呦。'

# 提取 mel 特征
front_end = Feat_extractor()

input_speech_resample = torchaudio.transforms.Resample(orig_freq=32000, new_freq=resample_rate)(input_speech_16k)
input_feat, input_feat_len = front_end(input_speech_resample)

# 提取 speaker embedding
embedding = extract_embedding(input_speech_16k_path, embedding_path)
embedding =  torch.tensor(embedding).unsqueeze(0).to('cuda')

In [28]:
# 生成语音和梅尔特征
with torch.no_grad():
    # 转换为张量
    tts_speech, tts_mel = audio_decoder.token2wav(
        token=audio_tokens,
        prompt_token = audio_tokens,
        uuid=this_uuid,
        prompt_feat=input_feat,
        embedding=embedding,
        finalize=True
    )

import IPython.display as ipd

# 假设 audio_data 是你的音频数据，sr 是采样率
# audio_data = np.array(new_dataset['mp3'][0]['array'])
sr = 22050
tts_speech = tts_speech.cpu()
# 在 Jupyter Notebook 中播放音频
ipd.Audio(tts_speech, rate=sr)